In [ ]:
from pathlib import Path
import sys
from hampel import hampel
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent  
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / Path("src")))

print(sys.path)

from src.moire.io import load_field, fmt4
from src.moire.draw_lines import plot_general_line, overlay_behaviors, overlay_features
from src.moire.extract_features import extract_downturns, extract_upturns
from src.moire.adaptive_multiscale_smooth import adaptive_multiscale_smooth
from src.moire.signal_helpers import local_noise

IN = ROOT / Path("source_data")
OUT = Path.cwd().resolve() / Path("output") / "debug_74mV"


In [6]:
T, nu, R = load_field(74, IN)

linecuts = []
for i, v in enumerate(nu):
    linecuts.append({"E": 74, "nu": v, "T": T, "rho": R[:, i]})

# Data Preprocessing
for linecut in linecuts:

    nu_interest = 1.377

    if linecut["nu"] < nu_interest - 0.05 or linecut["nu"] > nu_interest + 0.05:
        continue

    # Smoothing
    rho = linecut.get("rho")
    rho_hampel = hampel(rho).filtered_data
    rho_smoothed = adaptive_multiscale_smooth(T, rho, z_threshold=10)
    linecut.update({"rho_smoothed": rho_smoothed})

    # Noise estimates
    noise = local_noise(T, rho, rho_smoothed)
    linecut.update({"local_noise": noise})

    # Upturn & downturn feature extraction
    features = []
    features += extract_upturns(T, linecut)
    features += extract_downturns(T, linecut)
    linecut.update({"features": features})
    linecut.update({"behaviors": []})



In [8]:

# Filter Linecuts by nu

nu_left, nu_right = 1.377-0.05, 1.377+0.05
linecuts_nu = []
for linecut in linecuts:
    linecuts_nu.append(linecut) if nu_left <= linecut["nu"] and linecut["nu"] <= nu_right else None 

num_linecuts = 10
selected_linecuts = np.linspace(0, len(linecuts_nu), num_linecuts, dtype = "int")

for i, linecut in enumerate(linecuts_nu):
    if i in selected_linecuts:
        param_string = "   ".join(f"{k} = {fmt4(v)}" for k, v in linecut.items() if k == "E" or k == "nu")

        fig, ax = plt.subplots(figsize = (8, 6), dpi = 250)
        plot_general_line(ax, T, linecut["rho"], title = param_string, error = linecut["local_noise"],
                          xlim = (0, 3), ylim = (50, 150))
        plot_general_line(ax, T, linecut["rho_smoothed"], color = "red")
        overlay_features(ax, linecut, filter = 0.5)
        plt.savefig(str(OUT / Path(param_string + ".png")))
        plt.close(fig)